# 20 PIBT：逐步抢格与优先级继承


本课按百科条目写。PIBT 每拍为所有车选下一步，不搜整条路径。调度台在最后。

---

## 1. 定义

**Priority Inheritance with Backtracking**（Okumura et al., 2022）：每拍为所有车选下一格。

1. 按优先级从高到低。
2. 候选格按离自己目标的启发排序（含原地）。
3. 若格上有尚未决策的车：把优先级 **借** 给它，让它先让开（继承，类似 OS 优先级反转）。
4. 让不开则试下一候选；全失败则等待。

时间每拍大约 \(O(k \log k)\) 量级（实现还含递归）。无最优。无向可换位场景常常成功。有向窄廊可死锁。不完全。

LaCAM（下一课）用 PIBT 当后继生成器。

---

## 2. 不变量

- 每拍结束 `decided` 每人一格，且两两不同。
- `forbidden` 禁止把对方推到自己脚下造成互换死锁。
- 到达目标后优先级不再涨（库：未到达每拍 +1，到达 +0），让还在路上的车逐渐变高优先。

---

## 3. 伪代码

```
每拍:
    decided ← {}
    function plan(a, forbidden):
        if a 已 decided: return
        for v in 邻居∪{pos[a]} 按 dist-to-goal 排序:
            if v in forbidden 或 v 已被别人 decided 占用: continue
            if 有车 b 现在在 v 且未 decided:
                pri[b] ← pri[a]+ε
                plan(b, forbidden ∪ {pos[a]})
            if v 仍空: decided[a]←v; return
        decided[a]←pos[a]
    for a in 优先级降序: plan(a, ∅)
    全体走到 decided[a]
```

---

## 4. 手算（一拍）

a 在 (0,0) 要去右，b 在 (1,0) 挡路、自己也要去右。a 先决策，想进 (1,0)，发现 b 未决策 → 继承：b 先走 (2,0)，然后 a 进 (1,0)。一拍两车都右移。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 6. 实现：单拍 plan


In [ ]:
from lib.maps import grid_open
from lib.pretty import draw_grid


def pibt_step(graph, pos, goals, pri, verbose=True):
    agents = list(pos)
    occupied_next = {}
    decided = {}

    def dist(a, node):
        ga = goals[a]
        return abs(node[0] - ga[0]) + abs(node[1] - ga[1])

    def plan(a, forbidden):
        if a in decided:
            return decided[a] is not None
        cands = [n for n, _ in graph.neighbors(pos[a])] + [pos[a]]
        cands.sort(key=lambda n: (0 if n == goals[a] else 1, dist(a, n), str(n)))
        if verbose:
            print(f"  {a} 候选", cands, "forbidden", forbidden)
        for n in cands:
            if n in forbidden or n in occupied_next:
                continue
            holder = next((b for b, p in pos.items() if b != a and p == n), None)
            if holder is not None and holder not in decided:
                pri[holder] = pri[a] + 0.01
                if verbose:
                    print(f"    继承: {holder} 先让开 {n}")
                plan(holder, forbidden | {pos[a]})
                if occupied_next.get(n) is not None:
                    continue
            if n in occupied_next:
                continue
            decided[a] = n
            occupied_next[n] = a
            return True
        decided[a] = pos[a]
        occupied_next[pos[a]] = a
        return False

    for a in sorted(agents, key=lambda x: pri[x], reverse=True):
        plan(a, set())
    if verbose:
        print("decided", decided)
    return decided


g = grid_open(4, 1)
print(draw_grid(4, 1, start=(0, 0), goal=(3, 0)))
pos = {"a": (0, 0), "b": (1, 0)}
goals = {"a": (3, 0), "b": (3, 0)}
pri = {"a": 2.0, "b": 1.0}
print("一拍", pibt_step(g, pos, goals, pri))


In [ ]:
from evals.scenarios import two_agent_tunnel
from lib.maps import directed_deadlock_map
from lib.algos.rules import pibt

g2, starts, goals = two_agent_tunnel()
r = pibt(g2, starts, goals, steps=80)
print("开网格", r["found"], r["soc"])
print("有向", pibt(directed_deadlock_map(), {"a": "L", "b": "R"}, {"a": "R", "b": "L"}, steps=20)["found"])


## 7. 逐行

| 行 | 作用 |
|----|------|
| 候选按离目标排序 | 贪心朝目标 |
| holder 未 decided | 优先级继承 + 递归 |
| `forbidden ∪ {pos[a]}` | 不许对方踏进自己脚 |
| 全失败则原地 | 等待 |

## 常见 bug

1. 没有 forbidden，两车互换占格死循环。
2. 已下发、不能后退的段让「让开」失败。
3. 两车同一目标格：后到的人会一直想挤进去。本例 b 的目标也是 (3,0)，教学只演示一拍。

## 条目小结

| 项目 | 内容 |
|------|------|
| 粒度 | 每拍 |
| 核心 | 继承 + 回溯候选 |
| 最优 | 否 |
| 下一课 | 用 PIBT 当联合搜索的后继生成器 |

## 练习

1. 用 OS 优先级反转类比：低优先占着高优先要的格，为何要借优先级？
2. forbidden 拿掉后，两车对头会怎样？
3. PIBT 成功不代表 SOC 最优。造一个「能到但绕远」的直觉例子。


In [ ]:
from lib.studio.widget import PlannerStudio
studio = PlannerStudio()
studio.show_multi(g2.to_studio(), r, "PIBT")
studio
